# 00 · Сбор и подготовка данных

Семь стадий: котировки и состав индекса с MOEX ISS → новости из Telegram → привязка новостей
к тикерам → ruBERT-эмбеддинги → таргет и ценовые признаки → матрица признаков узлов →
динамический граф. Каждая стадия читает и пишет файлы в `data/`, поэтому стадии можно
перезапускать по отдельности.

В репозитории все выходные файлы уже лежат в `data/`, и этот ноутбук **не перезапускается**:
стадии 1–2 требуют сетевого доступа (MOEX ISS, Telegram API с ключами из `.env`), стадия 4
требует GPU/MPS (≈30 минут). Детерминированные стадии 3, 5, 6, 7 сверены с кэшем побайтово
(см. `Guide.md`).

In [ ]:
import re, os, time, bisect, itertools
from pathlib import Path
import numpy as np, pandas as pd

DATA = Path('data'); DATA.mkdir(exist_ok=True)

## 1. Состав IMOEX и дневные котировки (MOEX ISS)

Открытый API без ключей. Сектор эмитента берётся из состава отраслевых индексов Мосбиржи.
Состав индекса фиксируется на момент выгрузки: это survivorship bias, он оговорён в работе.

In [ ]:
import requests
BASE = 'https://iss.moex.com/iss'
SECTOR_INDICES = {'MOEXOG': 'Oil & Gas', 'MOEXEU': 'Electric Utilities', 'MOEXMM': 'Metals & Mining',
                  'MOEXTL': 'Telecom', 'MOEXFN': 'Financials', 'MOEXCN': 'Consumer', 'MOEXCH': 'Chemicals',
                  'MOEXRE': 'Real Estate', 'MOEXINN': 'IT & Innovation', 'MOEXTR': 'Transport'}
HISTORY_FROM, HISTORY_TILL = '2022-08-01', '2025-12-31'
HISTORY_COLS = 'TRADEDATE,OPEN,HIGH,LOW,CLOSE,VOLUME,VALUE'


def get(url, params, retries=3):
    for attempt in range(retries):                      # MOEX иногда рвёт соединение, повторяем с паузой
        try:
            r = requests.get(url, params=params, timeout=15); r.raise_for_status(); return r.json()
        except Exception:
            if attempt == retries - 1: raise
            time.sleep(2 ** attempt)


def index_members(index_id):
    js = get(f'{BASE}/statistics/engines/stock/markets/index/analytics/{index_id}.json',
             {'iss.meta': 'off', 'analytics.columns': 'ticker', 'limit': 100})
    return [r[0] for r in js.get('analytics', {}).get('data', []) if r[0]]


def fetch_history(secid):
    url = f'{BASE}/history/engines/stock/markets/shares/boards/TQBR/securities/{secid}.json'
    rows, start = [], 0
    while True:                                         # пагинация через history.cursor
        js = get(url, {'iss.meta': 'off', 'history.columns': HISTORY_COLS,
                       'from': HISTORY_FROM, 'till': HISTORY_TILL, 'start': start})
        chunk = js['history']['data']
        if not chunk: break
        rows += chunk
        index, total, pagesize = js['history.cursor']['data'][0]
        if index + pagesize >= total: break
        start = index + pagesize; time.sleep(0.3)
    return pd.DataFrame(rows, columns=HISTORY_COLS.split(',')).assign(SECID=secid)


js = get(f'{BASE}/engines/stock/markets/shares/boards/TQBR/securities.json',
         {'iss.meta': 'off', 'securities.columns': 'SECID,SHORTNAME,ISIN,LOTSIZE,FACEVALUE,PREVPRICE'})
tickers = pd.DataFrame(js['securities']['data'], columns=js['securities']['columns'])
sector = {}
for index_id, name in SECTOR_INDICES.items():
    for t in index_members(index_id):
        sector.setdefault(t, name)
tickers = tickers[tickers['SECID'].isin(index_members('IMOEX'))].reset_index(drop=True)
tickers['SECTOR'] = tickers['SECID'].map(sector).fillna('Other')
tickers.to_csv(DATA / 'tickers.csv', index=False)

history = pd.concat([fetch_history(s) for s in tickers['SECID']], ignore_index=True)
history['TRADEDATE'] = pd.to_datetime(history['TRADEDATE'])
history.sort_values(['SECID', 'TRADEDATE']).to_csv(DATA / 'history_ohlcv.csv', index=False)
print(f'{len(tickers)} тикеров, {len(history):,} строк котировок')

## 2. Новостной корпус Telegram

Три канала разных жанров: брокер (`bcs_express`), медиа (`RBCinvest`), агрегатор
(`smartlabnews`). Учётные данные читаются из `.env` (шаблон `.env.example`); файл сессии
Telethon в репозиторий не попадает. Сообщения идут от новых к старым, цикл обрывается
на первой дате раньше начала периода.

In [ ]:
from telethon import TelegramClient

if Path('.env').exists():                                # TELEGRAM_API_ID / TELEGRAM_API_HASH / TELEGRAM_PHONE
    for line in Path('.env').read_text().splitlines():
        k, _, v = line.partition('=')
        if k.strip() and not k.startswith('#'):
            os.environ.setdefault(k.strip(), v.strip())
CHANNELS = ['bcs_express', 'RBCinvest', 'smartlabnews']
START, END = pd.Timestamp('2022-08-01', tz='UTC'), pd.Timestamp('2026-01-01', tz='UTC')


async def fetch(client, channel):
    rows = []
    async for m in client.iter_messages(channel, offset_date=END):
        if m.date < START: break
        if m.text:
            rows.append(dict(channel=channel, date=m.date, text=m.text, views=m.views, forwards=m.forwards))
    return rows


async with TelegramClient(os.getenv('TELEGRAM_SESSION_NAME', 'news_session'),
                          int(os.environ['TELEGRAM_API_ID']), os.environ['TELEGRAM_API_HASH']) as client:
    await client.start(phone=os.environ['TELEGRAM_PHONE'])
    news = pd.DataFrame([r for c in CHANNELS for r in await fetch(client, c)])
news.to_csv(DATA / 'news_aug2022_dec2025.csv', index=False)
print(news['channel'].value_counts())

## 3. Привязка новостей к тикерам

Дедупликация репостов по нормализованным первым 200 символам, затем трёхуровневый каскад:
явное упоминание тикера или синонима (вес 1,0) → отраслевое ключевое слово, всем тикерам
сектора (0,5) → иначе всему индексу (0,2). Уровни взаимоисключающие. Регулярные выражения
используют границу слова и negative lookahead, чтобы «сбер» не ловил «сберегательный».

In [ ]:
news = pd.read_csv(DATA / 'news_aug2022_dec2025.csv', parse_dates=['date'])
tickers = pd.read_csv(DATA / 'tickers.csv')

key = news['text'].map(lambda t: ' '.join(t.lower().split())[:200] if isinstance(t, str) else '')
news = (news.assign(_k=key).sort_values('date').drop_duplicates('_k', keep='first')
            .drop(columns='_k').reset_index(drop=True))
news['news_id'] = news.index
print(f'после дедупликации: {len(news):,} новостей')

SYNONYMS = {
    'SBER': ['сбербанк', 'сбер(?!ег)', 'sber'], 'VTBR': ['втб', 'vtb'],
    'TCSG': ['тинькофф', 'т-банк', 'tbank', 'tinkoff'], 'MOEX': ['московская биржа', 'мосбиржа', 'moex'],
    'SFIN': ['сфп', 'sfin', 'эсэфай'], 'BSPB': ['банк санкт.петербург', 'bspb'],
    'CBOM': ['мкб', 'московский кредитный банк'],
    'LKOH': ['лукойл', 'lukoil'], 'GAZP': ['газпром', 'gazprom'], 'NVTK': ['новатэк', 'novatek'],
    'ROSN': ['роснефть', 'rosneft'], 'TATN': ['татнефть', 'tatneft'], 'SNGS': ['сургутнефтегаз', 'сургут(?!ск)'],
    'BANE': ['башнефть'], 'RNFT': ['русснефть'],
    'GMKN': ['норникель', 'норильский никель', 'norilsk nickel', 'gmkn'], 'NLMK': ['нлмк', 'nlmk'],
    'CHMF': ['северсталь', 'severstal'], 'MAGN': ['ммк', 'магнитогорск'], 'ALRS': ['алроса', 'alrosa'],
    'POLY': ['полиметалл', 'polymetal'], 'PLZL': ['полюс', 'polyus'], 'RUAL': ['русал', 'rusal'],
    'ENPG': ['эн\\+', 'en\\+'], 'RASP': ['распадская'], 'MTLR': ['мечел'],
    'FEES': ['фск', 'россети(?! мос)', 'rosseti'], 'HYDR': ['русгидро', 'rushydro'], 'IRAO': ['интер рао', 'inter rao'],
    'MSNG': ['мосэнерго'], 'OGKB': ['огк.2', 'ogk'], 'TGKA': ['тгк.1'],
    'MTSS': ['мтс(?! банк)', 'mts'], 'RTKM': ['ростелеком', 'rostelecom'], 'VKCO': ['вконтакте', '\\bвк\\b', '\\bvk\\b'],
    'HHRU': ['хедхантер', 'headhunter', 'hh\\.ru'], 'OZON': ['озон', 'ozon'], 'POSI': ['позитив', 'positive technolog'],
    'ASTR': ['астра', 'группа астра'], 'DIAS': ['диасофт'], 'WUSH': ['whoosh', 'вуш'], 'YDEX': ['яндекс', 'yandex'],
    'MGNT': ['магнит(?! ма)', 'magnit'], 'FIVE': ['x5', 'икс5', 'пятёрочка', 'перекрёсток'], 'LENT': ['лента'],
    'FIXP': ['fix price', 'фикс прайс'], 'BELU': ['белуга'], 'ABRD': ['абрау'],
    'AFLT': ['аэрофлот', 'aeroflot'], 'FESH': ['двмп', 'fesco'], 'NMTP': ['нмтп', 'новороссийск'], 'FLOT': ['совкомфлот', 'sovcomflot'],
    'SMLT': ['самолёт', 'samolet'], 'PIKK': ['пик(?! а)', 'pik group'], 'LSRG': ['лср', 'lsr'], 'ETLN': ['эталон', 'etalon'],
    'PHOR': ['фосагро', 'phosagro'], 'AKRN': ['акрон', 'acron'], 'KAZT': ['куйбышевазот'],
    'AGRO': ['русагро', 'rusagro'], 'SGZH': ['сегежа', 'segezha'], 'UWGN': ['унвл', 'объединённая вагонная'],
    'GLTR': ['globaltrans', 'глобалтранс'],
}
SECTOR_KEYWORDS = {
    'Oil & Gas': ['нефт', '\\bгаз\\b', 'нефтегаз', 'опек', '\\bopec\\b', 'нефтедобыч'],
    'Metals & Mining': ['металлург', 'сталелит', 'руда', 'золотодоб', 'никел', 'алюмини', 'добыч.{0,5}мет'],
    'Financials': ['банковский сектор', 'банк(?:и|ам|ов)\\b', 'кредитн', 'банкир'],
    'Electric Utilities': ['электроэнерг', 'энергетик', 'генерирующ'],
    'Telecom': ['телеком', 'оператор связи', 'сотовой связи'],
    'IT & Innovation': ['ит-сектор', 'айти', 'хайтек', 'софтверн', 'технологический сектор'],
    'Consumer': ['ритейл', 'розничн.{0,3}торгов', 'продуктов.{0,3}ритейл', 'fmcg'],
    'Real Estate': ['девелопер', 'жилищ', 'застройщик', 'недвижим'],
    'Chemicals': ['химическ', 'удобрени'],
    'Transport': ['авиаперевоз', 'грузоперевоз', 'портов'],
}
imoex = set(tickers['SECID'])
TICKER_RE = {t: re.compile('|'.join(p), re.I) for t, p in SYNONYMS.items() if t in imoex}
SECTOR_RE = {s: re.compile('|'.join(p), re.I) for s, p in SECTOR_KEYWORDS.items()}
sector_tickers = tickers.groupby('SECTOR')['SECID'].apply(list).to_dict()


def link(news_id, text):
    if not isinstance(text, str):
        return []
    hit = [t for t, r in TICKER_RE.items() if r.search(text)]
    if hit:
        return [(news_id, t, 1.0, 1) for t in hit]
    hit = dict.fromkeys(t for s, r in SECTOR_RE.items() if r.search(text) for t in sector_tickers.get(s, []))
    if hit:
        return [(news_id, t, 0.5, 2) for t in hit]
    return [(news_id, t, 0.2, 3) for t in tickers['SECID']]


linked = pd.DataFrame([row for i, t in zip(news['news_id'], news['text']) for row in link(i, t)],
                      columns=['news_id', 'SECID', 'weight', 'level'])
linked = (linked.merge(news[['news_id', 'date', 'channel', 'text', 'views', 'forwards']], on='news_id')
                .sort_values(['SECID', 'date']).reset_index(drop=True))
linked.to_csv(DATA / 'news_linked.csv', index=False)
print(linked.groupby('level').agg(связей=('news_id', 'size'), новостей=('news_id', 'nunique')))

## 4. ruBERT-эмбеддинги

`DeepPavlov/rubert-base-cased`, mean-pooling по последнему скрытому слою с учётом маски,
768 компонент. Считается один раз на уникальную новость и кэшируется; агрегация по узлам
делается на следующей стадии.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL = 'DeepPavlov/rubert-base-cased'
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
tok, model = AutoTokenizer.from_pretrained(MODEL), AutoModel.from_pretrained(MODEL).to(device).eval()

uniq = (pd.read_csv(DATA / 'news_linked.csv', usecols=['news_id', 'text', 'date', 'channel'])
          .drop_duplicates('news_id').sort_values('news_id').reset_index(drop=True))
uniq['text'] = uniq['text'].fillna('').astype(str)


@torch.no_grad()
def embed(texts):
    enc = tok(texts, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
    mask = enc['attention_mask'].unsqueeze(-1).float()
    h = model(**enc).last_hidden_state
    return ((h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)).cpu().numpy().astype(np.float32)


emb = np.vstack([embed(uniq['text'][i:i + 32].tolist()) for i in range(0, len(uniq), 32)])
np.save(DATA / 'news_embeddings.npy', emb)
uniq[['news_id', 'date', 'channel']].to_csv(DATA / 'news_embeddings_index.csv', index=False)
print(emb.shape)

## 5. Таргет и ценовые признаки

`FUTURE_RETURN(t)` определяется как простая доходность следующего торгового дня; трёхклассовая разметка по
терцилям делается позже, внутри каждого фолда, только по его train-части. Ценовые признаки:
пять лагов доходности и логарифм отношения объёма к его 20-дневному среднему.

In [ ]:
df = pd.read_csv(DATA / 'history_ohlcv.csv', parse_dates=['TRADEDATE']).sort_values(['SECID', 'TRADEDATE']).reset_index(drop=True)
g = df.groupby('SECID')
df['RETURN'] = g['CLOSE'].pct_change(fill_method=None)
for k in range(1, 6):
    df[f'LAG_RET_{k}'] = df.groupby('SECID')['RETURN'].shift(k)
vol_ma = g['VOLUME'].transform(lambda x: x.rolling(20, min_periods=5).mean())
df['LOG_VOL_NORM'] = np.log(df['VOLUME'].clip(lower=1) / vol_ma.clip(lower=1))
df['FUTURE_RETURN'] = df.groupby('SECID')['RETURN'].shift(-1)
df['TARGET'] = (df['FUTURE_RETURN'] > 0).astype(float).where(df['FUTURE_RETURN'].notna())
df = df.dropna(subset=['TARGET', 'RETURN', 'FUTURE_RETURN', 'LAG_RET_5', 'LOG_VOL_NORM']).copy()
df['TARGET'] = df['TARGET'].astype(int)                 # бинарный знак: унаследованная колонка, в моделях не используется
cols = ['SECID', 'TRADEDATE', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME', 'RETURN',
        'LAG_RET_1', 'LAG_RET_2', 'LAG_RET_3', 'LAG_RET_4', 'LAG_RET_5', 'LOG_VOL_NORM', 'FUTURE_RETURN', 'TARGET']
df[cols].reset_index(drop=True).to_csv(DATA / 'targets.csv', index=False)
print(f'{len(df):,} строк, {df.SECID.nunique()} тикеров')

## 6. Матрица признаков узлов (N × 775)

Узел: пара (тикер, торговый день). Время новостей переводится из UTC в Москву; новости после
закрытия аукциона 18:50 относятся к следующей торговой сессии. Эмбеддинг узла равен взвешенному
среднему эмбеддингов привязанных новостей (веса уровней 1,0 / 0,5 / 0,2), считается через
`np.add.at` чанками, чтобы не материализовать матрицу 482 205 × 768.

In [ ]:
PRICE_COLS = ['LAG_RET_1', 'LAG_RET_2', 'LAG_RET_3', 'LAG_RET_4', 'LAG_RET_5', 'LOG_VOL_NORM']
CLOSE_MSK = pd.Timestamp('18:50').time()

links = pd.read_csv(DATA / 'news_linked.csv', usecols=['news_id', 'SECID', 'weight', 'date'], parse_dates=['date'])
targets = pd.read_csv(DATA / 'targets.csv', parse_dates=['TRADEDATE'])
emb = np.load(DATA / 'news_embeddings.npy')
emb_pos = pd.Series(np.arange(len(emb)), index=pd.read_csv(DATA / 'news_embeddings_index.csv')['news_id'].values)

msk = links['date'].dt.tz_convert('Europe/Moscow') if links['date'].dt.tz is not None else links['date'].dt.tz_localize('UTC').dt.tz_convert('Europe/Moscow')
links['DATE'] = msk.dt.normalize().dt.tz_localize(None)
days = sorted(targets['TRADEDATE'].unique())
late = msk.dt.time >= CLOSE_MSK
links.loc[late, 'DATE'] = links.loc[late, 'DATE'].map(lambda d: days[i] if (i := bisect.bisect_right(days, d)) < len(days) else pd.NaT)
links = links.dropna(subset=['DATE']).sort_values(['SECID', 'DATE']).reset_index(drop=True)

grp = links.groupby(['SECID', 'DATE'], sort=False)                 # узел = (тикер, день)
gid = grp.ngroup().to_numpy()
keys = grp.size().reset_index()[['SECID', 'DATE']].assign(emb_row=np.arange(grp.ngroups))
w, rows = links['weight'].to_numpy(np.float32), emb_pos.loc[links['news_id']].to_numpy()
sum_e, sum_w = np.zeros((grp.ngroups, emb.shape[1]), np.float32), np.zeros(grp.ngroups, np.float32)
for s in range(0, len(links), 100_000):
    sl = slice(s, s + 100_000)
    np.add.at(sum_e, gid[sl], emb[rows[sl]] * w[sl, None]); np.add.at(sum_w, gid[sl], w[sl])
mean_e = sum_e / np.maximum(sum_w[:, None], 1e-12)

base = targets.dropna(subset=PRICE_COLS + ['FUTURE_RETURN', 'TARGET']).reset_index(drop=True)
m = base.merge(keys, left_on=['SECID', 'TRADEDATE'], right_on=['SECID', 'DATE'], how='left')
has = m['emb_row'].notna().to_numpy()
X = np.zeros((len(m), len(PRICE_COLS) + 1 + emb.shape[1]), np.float32)
X[:, :6], X[:, 6] = m[PRICE_COLS].to_numpy(np.float32), has
X[has, 7:] = mean_e[m.loc[has, 'emb_row'].to_numpy(int)]
assert np.isfinite(X).all()
np.save(DATA / 'node_features.npy', X)
m[['SECID', 'TRADEDATE', 'FUTURE_RETURN', 'TARGET']].assign(has_news=has.astype(np.int8)).to_csv(DATA / 'node_features_index.csv', index=False)
print(X.shape, f'has_news = 1 на {has.mean():.1%} строк')

## 7. Динамический граф эмитентов

Минимальное остовное дерево (Mantegna, 1999) на 60-дневном окне корреляций log-доходностей,
из которых вычтено кросс-секционное среднее дня: без этого почти все пары положительно
скоррелированы и дерево вырождается в звезду. Расстояние берётся по |ρ|, знак хранится как
атрибут ребра `[sign ρ, |ρ|]`. Граф считается на последний день месяца и действует весь
следующий месяц. Отдельно строится статический секторный граф для абляции.

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree

WINDOW, MIN_OBS = 60, 40
ohlcv = pd.read_csv(DATA / 'history_ohlcv.csv', parse_dates=['TRADEDATE'])
nfi = pd.read_csv(DATA / 'node_features_index.csv', parse_dates=['TRADEDATE'])
tickers = pd.read_csv(DATA / 'tickers.csv')
nodes = (tickers[tickers['SECID'].isin(nfi['SECID'].unique())][['SECID', 'SHORTNAME', 'SECTOR']]
         .drop_duplicates('SECID').sort_values('SECID').reset_index(drop=True))
nodes.insert(0, 'node_id', nodes.index)
nid = dict(zip(nodes['SECID'], nodes['node_id']))

close = ohlcv.pivot(index='TRADEDATE', columns='SECID', values='CLOSE').sort_index().reindex(columns=nodes['SECID'])
ret = np.log(close / close.shift(1)).iloc[1:]
ret = ret.sub(ret.mean(axis=1), axis=0)                          # убираем общерыночную моду
month_ends = pd.Series(ret.index, index=ret.index).groupby(ret.index.to_period('M')).max()


def mst_edges(window):
    valid = window.columns[window.notna().sum() >= MIN_OBS]
    if len(valid) < 2:
        return [], 0
    rho = np.nan_to_num(window[valid].corr().values); np.fill_diagonal(rho, 1.0)
    dist = np.sqrt(np.clip(2 * (1 - np.abs(rho)), 0, None))
    mst = minimum_spanning_tree(csr_matrix(np.triu(dist + 1e-9, k=1))).tocoo()
    edges = []
    for u, v in zip(mst.row, mst.col):
        a, b = sorted((nid[valid[u]], nid[valid[v]]))
        r = float(rho[u, v]); edges.append((a, b, int(np.sign(r)) if r != 0 else 1, abs(r)))
    return edges, len(valid)


graphs, meta = {}, []
for me in month_ends:
    p = ret.index.get_loc(me)
    edges, n_active = mst_edges(ret.iloc[max(0, p - WINDOW + 1):p + 1])
    if not edges:
        continue
    bucket = (me.to_period('M') + 1).strftime('%Y-%m')          # действует на следующий месяц
    graphs[bucket] = edges
    meta.append(dict(month_bucket=bucket, window_end=me.date(), n_active_nodes=n_active, n_edges=len(edges),
                     n_pos=sum(e[2] > 0 for e in edges), n_neg=sum(e[2] < 0 for e in edges),
                     mean_abs_corr=np.mean([e[3] for e in edges]), median_abs_corr=np.median([e[3] for e in edges])))
meta = pd.DataFrame(meta)
assert (meta['n_edges'] == meta['n_active_nodes'] - 1).all()      # дерево: рёбер = узлов − 1

nodes.to_csv(DATA / 'graph_nodes.csv', index=False)
pd.DataFrame([(mb, *e) for mb, es in graphs.items() for e in es],
             columns=['month_bucket', 'src', 'dst', 'sign', 'abs_corr']).sort_values(['month_bucket', 'src', 'dst']).to_csv(DATA / 'graph_edges_corr.csv', index=False)
ei, ea = {}, {}
for mb, es in graphs.items():                                     # PyG-формат: обе ориентации, [прямые; обратные]
    src, dst, sg, ac = map(np.array, zip(*es))
    ei[mb] = np.stack([np.r_[src, dst], np.r_[dst, src]]).astype(np.int64)
    ea[mb] = np.stack([np.r_[sg, sg], np.r_[ac, ac]], 1).astype(np.float32)
np.savez(DATA / 'edge_index_corr.npz', **ei); np.savez(DATA / 'edge_attr_corr.npz', **ea)
meta.to_csv(DATA / 'graph_meta.csv', index=False)

sec = pd.DataFrame([dict(src=a, dst=b, sector=s) for s, g in nodes[nodes['SECTOR'] != 'Other'].groupby('SECTOR')
                    for a, b in itertools.combinations(sorted(g['node_id']), 2)])
sec.to_csv(DATA / 'graph_edges_sector.csv', index=False)
np.save(DATA / 'edge_index_sector.npy', np.stack([np.r_[sec.src, sec.dst], np.r_[sec.dst, sec.src]]).astype(np.int64))

dates = pd.DatetimeIndex(sorted(nfi['TRADEDATE'].unique()))
first = min(graphs)
pd.DataFrame({'TRADEDATE': dates.strftime('%Y-%m-%d'),
              'month_bucket': [m if (m := d.strftime('%Y-%m')) in graphs else first for d in dates]}).to_csv(DATA / 'date_to_month.csv', index=False)
print(f'{len(graphs)} месячных графов, доля отрицательных рёбер {meta.n_neg.sum() / meta.n_edges.sum():.3f}, '
      f'секторных рёбер {len(sec)}')